# The Molecular Machine Lab: Simulating Hinge Motions

This tutorial demonstrates how to simulate conformational transitions in a multi-domain protein or a simple peptide hinge using `synth-pdb`'s interpolation feature. We will generate two extreme states ('Open' and 'Closed') and compute the path between them. Then we will simulate how Circular Dichroism (CD) spectra and NMR chemical shifts change dynamically along the transition path.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # @title Environment Setup
    !pip install -q synth-pdb biotite matplotlib numpy scipy py3Dmol openmm
else:
    sys.path.append(os.path.abspath("../../"))

print("✅ Environment configured!")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import biotite.structure.io.pdb as pdb
from synth_pdb.generator import generate_pdb_content
from synth_pdb.quality.interpolate import interpolate_structures
from synth_pdb.cd_simulator import CDSimulator
import tempfile

# Create a directory to store our trajectory
os.makedirs("hinge_trajectory", exist_ok=True)

## 1. Generating the 'Open' and 'Closed' States

We will design a simple 30-residue sequence. 
- **State A (Open)**: Two alpha helices connected by an extended random loop.
- **State B (Closed)**: Two alpha helices connected by a beta turn (forming a hairpin shape).

In [ ]:
sequence = "AEQLLKALEQA" + "GGSGG" + "AEQLLKALEQA"

# State A: Open Hinge
open_spec = "1-11:alpha,12-16:extended,17-27:alpha"
open_pdb = generate_pdb_content(
    sequence_str=sequence,
    conformation="alpha",  # fallback
    structure=open_spec,
    minimize_energy=True,
)
with open("hinge_trajectory/open_state.pdb", "w") as f:
    f.write(open_pdb)

# State B: Closed Hinge
closed_spec = "1-11:alpha,12-16:random,17-27:alpha"  # random for flexibility, or we can use beta turn, let's use beta-turn if available or just random
closed_pdb = generate_pdb_content(
    sequence_str=sequence, conformation="alpha", structure=closed_spec, minimize_energy=True
)
with open("hinge_trajectory/closed_state.pdb", "w") as f:
    f.write(closed_pdb)

print("Generated Open and Closed states.")

## 2. Interpolating the Trajectory

Using `interpolate_structures`, we can morph the backbone torsion angles between the open and closed states.

In [ ]:
steps = 10
output_prefix = "hinge_trajectory/morph"

# Perform linear interpolation in torsion space
interpolate_structures(
    "hinge_trajectory/open_state.pdb",
    "hinge_trajectory/closed_state.pdb",
    steps=steps,
    output_prefix=output_prefix,
)
print("Trajectory generation complete.")

## 3. Simulating Biophysical Observables Along the Path

Let's calculate the Circular Dichroism (CD) spectra for each frame to see how the secondary structure signature changes during the hinge motion.

In [ ]:
plt.figure(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, steps + 1))

for i in range(steps + 1):
    frame_path = f"{output_prefix}_{i}.pdb"
    pdb_file = pdb.PDBFile.read(frame_path)
    structure = pdb_file.get_structure(model=1)

    cd_sim = CDSimulator(structure)
    spectrum = cd_sim.get_spectrum(noise_level=0)  # remove noise for clear trend

    plt.plot(np.arange(190, 251, 1), spectrum, color=colors[i], alpha=0.7, label=f"Frame {i}")

plt.title("Dynamic CD Spectra during Hinge Motion")
plt.xlabel("Wavelength (nm)")
plt.ylabel("Molar Ellipticity")
plt.grid(alpha=0.3)
plt.show()